# First-principles proof for the fire life cycle

This notebook derives every time-series line in manuscript Figure 4 from the stated two-thirds metabolic closure. It proves the analytical identities symbolically, checks invariant bounds, reproduces the numerical trajectory, and verifies the implementation to machine precision.

## Evidence boundary

This is a proof of the consequences of the named assumptions. It is not a proof that real wildfire perimeters have dimension $4/3$, that real fires obey the closure, or that the selected connectivity equation is unique. Those remain empirical hypotheses. Here, *first principles* means that geometry, kinematics, finite fuel, transport matching, and mass-action connectivity are declared once and every plotted signal is derived from them.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp

from fire_metabolism.manuscript_figures import (
    LifeCycleParameters,
    run_life_cycle_validation,
    simulate_fire_life_cycle,
)

## 1. Geometry fixes the metabolic exponent

Let $L$ be a characteristic footprint length. The burned area is two-dimensional, $A\propto L^2$. Under the manuscript hypothesis that the projected external hull has dimension $D_h=4/3$, perimeter scales as $P\propto L^{D_h}$. Eliminating $L$ gives

$$P\propto A^{D_h/2}=A^{2/3}.$$

Thus the $2/3$ exponent is not fitted inside Figure 4. It is the conditional consequence of assuming $D_h=4/3$.

In [ ]:
D_h = sp.Rational(4, 3)
sigma = sp.simplify(D_h / 2)
constant_beta_time_exponent = sp.simplify(1 / (1 - sigma))
perimeter_time_exponent = sp.simplify(sigma / (1 - sigma))
assert sigma == sp.Rational(2, 3)
assert constant_beta_time_exponent == 3
assert perimeter_time_exponent == 2
{
    'perimeter_area_exponent': sigma,
    'constant_beta_area_time_exponent': constant_beta_time_exponent,
    'constant_beta_perimeter_time_exponent': perimeter_time_exponent,
}

## 2. Boundary kinematics gives the growth law

An active boundary element $d\ell$ advancing normally at speed $v_n$ creates area $dA=v_n\,d\ell\,dt$. Integrating over the coherent active boundary gives $dA/dt=v_{eff}P_a$. Connectivity $C\in[0,1]$ determines the coherent fraction of the geometric boundary, so

$$P_a=kCA^{2/3}.$$

Let remaining connected fuel be $F$ and transport-matching efficiency be $\eta$. With $v_{eff}=v_0F\eta$ and $\beta_0=v_0k$,

$$\frac{dA}{dt}=v_{eff}P_a=\beta_0CF\eta A^{2/3}.$$


In [ ]:
A, C, F, eta, k, v0 = sp.symbols('A C F eta k v_0', positive=True)
P_active = k * C * A ** sp.Rational(2, 3)
v_eff = v0 * F * eta
beta0 = v0 * k
metabolic_rate = beta0 * C * F * eta * A ** sp.Rational(2, 3)
assert sp.simplify(v_eff * P_active - metabolic_rate) == 0

# Cube-root area removes the two-thirds state dependence exactly.
cube_root_rate = sp.simplify(sp.diff(A ** sp.Rational(1, 3), A) * metabolic_rate)
assert cube_root_rate == beta0 * C * F * eta / 3
cube_root_rate

The transformed state $X=A^{1/3}$ therefore satisfies

$$\frac{dX}{dt}=\frac{\beta_0}{3}CF\eta.$$

If $CF\eta$ is constant, $X$ is linear, $A\propto t^3$, and $P\propto t^2$. Figure 4 does **not** assume a constant prefactor: fuel depletion and connectivity dynamics make $CF\eta$ rise and fall, producing acceleration and death while preserving the $A^{2/3}$ metabolic factor.

## 3. Finite fuel and impedance matching

The connected fuel fraction is bookkeeping, not an independently drawn forcing curve:

$$F(A)=\frac{A_{max}-A}{A_{max}-A_0}.$$

Supply and removal are compared with $r=C/F$. The matching function is

$$\eta(r)=\frac{4r}{(1+r)^2}.$$

The identity $1-\eta=[(1-r)/(1+r)]^2$ proves $0<\eta\leq1$, with equality only at $r=1$.

In [ ]:
r = sp.symbols('r', positive=True)
eta_r = 4 * r / (1 + r) ** 2
matching_identity = sp.factor(1 - eta_r)
assert sp.simplify(matching_identity - ((1 - r) / (1 + r)) ** 2) == 0
assert sp.simplify(eta_r.subs(r, 1)) == 1
assert sp.simplify(sp.diff(eta_r, r).subs(r, 1)) == 0
assert sp.diff(eta_r, r, 2).subs(r, 1) < 0
matching_identity

## 4. Connectivity is recruited and lost

Coherent connectivity follows a minimal mass-action balance:

$$\frac{dC}{dt}=\alpha C(1-C)F-\mu C.$$

Recruitment requires an existing connected edge, unused network capacity, and connected fuel. Fragmentation removes connectivity at rate $\mu C$. The interval $C\in[0,1]$ is invariant: at $C=0$, $dC/dt=0$; at $C=1$, $dC/dt=-\mu<0$. For fixed $F$, the nonzero equilibrium is $C^*=1-\mu/(\alpha F)$, which disappears when $F\leq\mu/\alpha$. Fuel exhaustion therefore forces eventual connectivity collapse.

In [ ]:
alpha, mu = sp.symbols('alpha mu', positive=True)
C_symbol = sp.symbols('C', nonnegative=True)
C_dot = alpha * C_symbol * (1 - C_symbol) * F - mu * C_symbol
assert sp.simplify(C_dot.subs(C_symbol, 0)) == 0
assert sp.simplify(C_dot.subs(C_symbol, 1)) == -mu
nonzero_equilibrium = sp.solve(sp.Eq(C_dot, 0), C_symbol)[1]
assert sp.simplify(nonzero_equilibrium - (1 - mu / (alpha * F))) == 0
nonzero_equilibrium

## 5. Why metabolic rate rises and then falls

Write $M=dA/dt=\beta_0CF\eta A^{2/3}$. Its logarithmic derivative is

$$\frac{1}{M}\frac{dM}{dt}=\frac{\dot C}{C}+\frac{\dot F}{F}+\frac{\dot\eta}{\eta}+\frac{2}{3}\frac{M}{A}.$$

The last term is the geometric acceleration contributed by $A^{2/3}$. Early connectivity recruitment makes the first term positive. Later, fuel depletion, loss of transport capacity, and fragmentation make the negative terms dominate. Peak growth occurs exactly when this sum crosses zero. No Gaussian or logistic metabolic-rate curve is supplied to the model.

In [ ]:
A_max, A_0 = sp.symbols('A_max A_0', positive=True)
M = sp.symbols('M', positive=True)
C_dot_symbol, eta_dot = sp.symbols('C_dot eta_dot', real=True)
F_dot = -M / (A_max - A_0)
log_metabolic_rate_change = (
    C_dot_symbol / C + F_dot / F + eta_dot / eta + sp.Rational(2, 3) * M / A
)
log_metabolic_rate_change

The active perimeter has a related but distinct peak condition:

$$\frac{1}{P_a}\frac{dP_a}{dt}=\frac{\dot C}{C}+\frac{2}{3}\frac{M}{A}.$$

Consequently, active perimeter can peak after metabolic rate: effective velocity and fuel can decline before geometric edge recruitment is fully lost.

## 6. Numerical reproduction and exact residual checks

The tested package integrates only $A$ and $C$. Fuel, perimeter, velocity, metabolic rate, energy release, fragmentation, and order parameters are derived afterward from the equations above.

In [ ]:
parameters = LifeCycleParameters()
trajectory = simulate_fire_life_cycle(parameters)
stage_table = pd.DataFrame(
    [
        {
            'stage': stage,
            'time_hours': trajectory.time[index],
            'area_ha': trajectory.area[index],
            'metabolic_rate_ha_per_hour': trajectory.growth_rate[index],
            'connectivity': trajectory.connectivity[index],
            'fuel_fraction': trajectory.fuel_fraction[index],
        }
        for stage, index in trajectory.stage_indices.items()
    ]
)
stage_table

In [ ]:
two_thirds_residual = trajectory.growth_rate - trajectory.beta * trajectory.area ** (2 / 3)
kinematic_residual = trajectory.growth_rate - trajectory.effective_velocity * trajectory.active_perimeter
perimeter_residual = trajectory.active_perimeter - (
    parameters.perimeter_coefficient * trajectory.connectivity * trajectory.area ** (2 / 3)
)
checks = {
    'max_abs_two_thirds_residual': float(np.max(np.abs(two_thirds_residual))),
    'max_abs_kinematic_residual': float(np.max(np.abs(kinematic_residual))),
    'max_abs_perimeter_residual': float(np.max(np.abs(perimeter_residual))),
    'area_is_monotone': bool(np.all(np.diff(trajectory.area) >= 0)),
    'area_stays_below_fuel_limit': bool(np.all(trajectory.area <= parameters.maximum_burnable_area_ha)),
    'connectivity_stays_bounded': bool(np.all((trajectory.connectivity >= 0) & (trajectory.connectivity <= 1))),
}
assert checks['max_abs_two_thirds_residual'] < 1e-12
assert checks['max_abs_kinematic_residual'] < 1e-11
assert checks['max_abs_perimeter_residual'] < 1e-12
assert checks['area_is_monotone']
assert checks['area_stays_below_fuel_limit']
assert checks['connectivity_stays_bounded']
checks

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True, constrained_layout=True)
t = trajectory.time
axes[0].plot(t, trajectory.area, color='#5B8EE6', lw=2.8, label='cumulative area A')
ax0r = axes[0].twinx()
ax0r.plot(t, trajectory.active_perimeter, color='#B52322', lw=2.3, ls='--', label='active perimeter P_a')
axes[0].set_ylabel('Area (ha)')
ax0r.set_ylabel('Active perimeter')
axes[1].plot(t, trajectory.growth_rate, color='#5B8EE6', lw=2.8, label='metabolic rate dA/dt')
axes[1].plot(t, trajectory.beta / parameters.beta_scale, color='#F2D45C', lw=2.3, label='C F eta')
axes[1].set_ylabel('Rate / normalized prefactor')
axes[1].legend()
axes[2].plot(t, trajectory.connectivity, color='#2E8B57', lw=2.6, label='connectivity C')
axes[2].plot(t, trajectory.fuel_fraction, color='#6B6F73', lw=2.2, ls='--', label='fuel F')
axes[2].plot(t, trajectory.coupling, color='#7B4BA3', lw=2.4, label='coupling capacity q')
axes[2].set_ylabel('State fraction')
axes[2].set_xlabel('Time since ignition (h)')
axes[2].legend(ncol=3)
for ax in axes:
    ax.grid(alpha=0.25)
fig.suptitle(r'All trajectories derived from $dA/dt=\beta_0CF\eta A^{2/3}$')

## 7. Robustness and prediction are separate questions

The ensemble perturbs fuel capacity, connectivity recruitment and loss, the metabolic coefficient, ignition size, and perimeter coefficient. Structural robustness asks whether cumulative area remains monotone and the stages remain ordered. Prediction compares four-hour forecasts using only observations available before each forecast origin.

In [ ]:
records, validation = run_life_cycle_validation(n_runs=300, seed=20260927)
validation_table = pd.DataFrame(validation['metrics'])
assert validation['monotonic_area_fraction'] == 1.0
assert validation['ordered_stage_fraction'] == 1.0
validation_table.pivot(
    index='phase',
    columns='model',
    values='median_absolute_percentage_error',
).round(2)

## Conclusion

Given the $D_h=4/3$ geometric hypothesis and the stated transport/connectivity closures, the two-thirds metabolic factor, cube-root transformation, bounded finite-fuel trajectory, rise and fall of metabolic rate, delayed active-perimeter peak, and connectivity collapse all follow. The numerical implementation satisfies the defining identities to floating-point precision. Empirical work must still test the assumptions, estimate parameters without leakage, and compare this model against simpler and more flexible alternatives on untouched fires.